In [ ]:
print(1)

In [ ]:
!pip install transformers accelerate albumentations wandb -q

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image, ImageDraw
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
from transformers import SegformerForSemanticSegmentation
import wandb


os.environ["WANDB_API_KEY"] = "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"
wandb.login(key="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")

BASE_DIR = "/kaggle/input/datasets/sounakp/idd-segmentation/IDD_Segmentation"
IMG_DIR  = BASE_DIR + "/leftImg8bit"
MASK_DIR = BASE_DIR + "/gtFine"
SAVE_DIR = "/kaggle/working/masks"

LABEL2ID = {
    'road': 0, 'parking': 1, 'drivable fallback': 2,
    'sidewalk': 3, 'rail track': 4, 'non-drivable fallback': 5,
    'person': 6, 'animal': 7, 'rider': 8, 'motorcycle': 9,
    'bicycle': 10, 'autorickshaw': 11, 'car': 12, 'truck': 13,
    'bus': 14, 'caravan': 15, 'vehicle fallback': 16, 'curb': 17,
    'wall': 18, 'fence': 19, 'guard rail': 20, 'billboard': 21,
    'traffic sign': 22, 'traffic light': 23, 'pole': 24,
    'obs-str-bar-fallback': 25, 'building': 26, 'bridge': 27,
    'tunnel': 28, 'vegetation': 29, 'sky': 30,
    'fallback background': 31, 'unlabeled': 255
}
ID2LABEL     = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES  = 33
IGNORE_INDEX = 32
IMG_SIZE     = (512, 512)
BATCH_SIZE   = 4
EPOCHS       = 15
LR           = 6e-5
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
def json_to_mask(json_path, height, width):
    mask = Image.fromarray(np.full((height, width), 255, dtype=np.uint8))
    draw = ImageDraw.Draw(mask)
    with open(json_path) as f:
        data = json.load(f)
    for obj in data["objects"]:
        poly = [(p[0], p[1]) for p in obj["polygon"]]
        if len(poly) < 3:
            continue
        draw.polygon(poly, fill=LABEL2ID.get(obj["label"], 255))
    return np.array(mask)

print("Pre-rendering masks...")
for split in ["train", "val"]:
    if not os.path.exists(MASK_DIR + f"/{split}"):
        continue
    for city in tqdm(os.listdir(MASK_DIR + f"/{split}"), desc=split):
        save_city = f"{SAVE_DIR}/{split}/{city}"
        os.makedirs(save_city, exist_ok=True)
        for fname in os.listdir(MASK_DIR + f"/{split}/{city}"):
            if not fname.endswith(".json"):
                continue
            json_path = MASK_DIR + f"/{split}/{city}/{fname}"
            save_name = fname.replace("_gtFine_polygons.json", "_mask.png")
            save_path = f"{save_city}/{save_name}"
            if os.path.exists(save_path):
                continue
            with open(json_path) as f:
                meta = json.load(f)
            mask = json_to_mask(json_path, meta["imgHeight"], meta["imgWidth"])
            Image.fromarray(mask).save(save_path)

print("Done.")
print("Train masks:", sum([len(f) for r,d,f in os.walk(SAVE_DIR + "/train")]))
print("Val masks:",   sum([len(f) for r,d,f in os.walk(SAVE_DIR + "/val")]))

In [ ]:
# Base transforms — no augmentation
base_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# Gambia-motivated augmentation pipeline
gambia_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    # High-luminance equatorial lighting
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    # Laterite road colouring — HSV shift toward red-brown
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.7),
    # Harmattan dust/haze
    A.GaussianBlur(blur_limit=(3,7), p=0.5),
    # Replace RandomFog
    A.RandomFog(fog_coef_range=(0.1, 0.3), alpha_coef=0.1, p=0.4),
    # Replace CoarseDropout
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(20, 40), hole_width_range=(20, 40), p=0.4),
    # Mixed-traffic scaling
    A.RandomScale(scale_limit=0.3, p=0.4),
    A.PadIfNeeded(IMG_SIZE[0], IMG_SIZE[1], border_mode=0),
    A.CenterCrop(IMG_SIZE[0], IMG_SIZE[1]),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

class IDDDataset(Dataset):
    def __init__(self, img_dir, mask_dir, split="train", transform=None):
        self.transform = transform
        self.pairs = []
        for city in os.listdir(f"{img_dir}/{split}"):
            img_city  = f"{img_dir}/{split}/{city}"
            mask_city = f"{mask_dir}/{split}/{city}"
            if not os.path.exists(mask_city):
                continue
            for fname in os.listdir(img_city):
                if not fname.endswith(".png"):
                    continue
                img_path  = f"{img_city}/{fname}"
                mask_name = fname.replace("_leftImg8bit.png", "_mask.png")
                mask_path = f"{mask_city}/{mask_name}"
                if os.path.exists(mask_path):
                    self.pairs.append((img_path, mask_path))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img  = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path))
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out["image"]
            mask = out["mask"].long()
        mask[mask == 255] = IGNORE_INDEX
        return img, mask

# Train datasets — one without aug, one with
train_dataset_base = IDDDataset(IMG_DIR, SAVE_DIR, "train", transform=base_transform)
train_dataset_aug  = IDDDataset(IMG_DIR, SAVE_DIR, "train", transform=gambia_transform)

# Val split → 50% val, 50% test (no masks in IDD test split)
full_val  = IDDDataset(IMG_DIR, SAVE_DIR, "val", transform=base_transform)
val_size  = len(full_val) // 2
test_size = len(full_val) - val_size
val_dataset, test_dataset = random_split(
    full_val, [val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader_base = DataLoader(train_dataset_base, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
train_loader_aug  = DataLoader(train_dataset_aug,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader        = DataLoader(val_dataset,         batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader       = DataLoader(test_dataset,        batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train:  {len(train_dataset_base)}")
print(f"Val:    {len(val_dataset)}")
print(f"Test:   {len(test_dataset)}")

# Sanity check
imgs, masks = next(iter(train_loader_base))
print(f"Image batch: {imgs.shape} | Mask batch: {masks.shape}")
print(f"Mask unique values: {masks.unique()}")

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 22))
for i in range(5):
    img, mask = val_dataset[i]
    img_show  = img.permute(1,2,0).numpy()
    img_show  = img_show * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
    img_show  = np.clip(img_show, 0, 1)
    axes[i][0].imshow(img_show);           axes[i][0].set_title(f"Image {i+1}"); axes[i][0].axis("off")
    axes[i][1].imshow(mask, cmap="tab20"); axes[i][1].set_title(f"Mask {i+1}");  axes[i][1].axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/sample_visualization.png", dpi=150)
plt.show()

In [ ]:
def load_model():
    m = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/mit-b2",
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True
    )
    return m.to(device)

def upsample_logits(logits, size=(512,512)):
    return F.interpolate(logits, size=size, mode="bilinear", align_corners=False)

def evaluate(model, loader):
    model.eval()
    total_loss   = 0
    all_miou, all_acc, all_f1 = [], [], []
    class_ious   = np.zeros(NUM_CLASSES - 1)
    class_counts = np.zeros(NUM_CLASSES - 1)

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs     = model(pixel_values=imgs, labels=masks)
            total_loss += outputs.loss.item()
            logits      = upsample_logits(outputs.logits)
            preds       = logits.argmax(dim=1).cpu().numpy()
            labels      = masks.cpu().numpy()

            for pred, label in zip(preds, labels):
                fp, fl = pred.flatten(), label.flatten()
                valid  = fl != IGNORE_INDEX
                fp, fl = fp[valid], fl[valid]
                if len(fl) == 0:
                    continue

                all_acc.append((fp == fl).sum() / len(fl))

                ious, f1s = [], []
                for cls in range(NUM_CLASSES - 1):
                    tp  = ((fp==cls) & (fl==cls)).sum()
                    fpp = ((fp==cls) & (fl!=cls)).sum()
                    fn  = ((fp!=cls) & (fl==cls)).sum()
                    if tp + fpp + fn == 0:
                        continue
                    iou = tp / (tp + fpp + fn)
                    f1  = (2*tp) / (2*tp + fpp + fn + 1e-8)
                    ious.append(iou)
                    f1s.append(f1)
                    class_ious[cls]   += iou
                    class_counts[cls] += 1

                if ious:
                    all_miou.append(np.mean(ious))
                    all_f1.append(np.mean(f1s))

    per_class = {
        ID2LABEL.get(c, str(c)): class_ious[c] / class_counts[c]
        for c in range(NUM_CLASSES - 1) if class_counts[c] > 0
    }
    return {
        "val_loss":      total_loss / len(loader),
        "mIoU":          float(np.mean(all_miou)),
        "pixel_acc":     float(np.mean(all_acc)),
        "mean_f1":       float(np.mean(all_f1)),
        "per_class_iou": per_class
    }

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        loss = model(pixel_values=imgs, labels=masks).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def run_experiment(run_name, train_loader, epochs=EPOCHS):
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {run_name}")
    print(f"{'='*60}")
    wandb.init(project="gambia-segmentation", name=run_name, reinit="finish_previous")

    model     = load_model()
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = PolynomialLR(optimizer, total_iters=epochs, power=1.0)
    best_miou = 0.0
    results   = []

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        metrics    = evaluate(model, val_loader)
        scheduler.step()

        wandb.log({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   metrics["val_loss"],
            "mIoU":       metrics["mIoU"],
            "pixel_acc":  metrics["pixel_acc"],
            "mean_f1":    metrics["mean_f1"],
            **{f"iou/{k}": v for k, v in metrics["per_class_iou"].items()}
        })

        print(f"Epoch {epoch+1:02d} | Loss: {train_loss:.4f} | Val Loss: {metrics['val_loss']:.4f} "
              f"| mIoU: {metrics['mIoU']:.4f} | Acc: {metrics['pixel_acc']:.4f} | F1: {metrics['mean_f1']:.4f}")

        if metrics["mIoU"] > best_miou:
            best_miou = metrics["mIoU"]
            torch.save(model.state_dict(), f"/kaggle/working/{run_name}_best.pt")
            print(f"  ✓ Saved (mIoU: {best_miou:.4f})")

        results.append(metrics)

    # Final evaluation on held-out test set
    print(f"\nEvaluating on test set...")
    test_metrics = evaluate(model, test_loader)
    print(f"Test mIoU: {test_metrics['mIoU']:.4f} | Test Acc: {test_metrics['pixel_acc']:.4f} | Test F1: {test_metrics['mean_f1']:.4f}")
    wandb.log({"test_mIoU": test_metrics["mIoU"], "test_acc": test_metrics["pixel_acc"], "test_f1": test_metrics["mean_f1"]})
    wandb.finish()

    return model, results, test_metrics

In [ ]:
print("PHASE 1: Baseline evaluation — no adaptation")
baseline_model   = load_model()
baseline_val     = evaluate(baseline_model, val_loader)
baseline_test    = evaluate(baseline_model, test_loader)

print(f"\nVal  — mIoU: {baseline_val['mIoU']:.4f} | Acc: {baseline_val['pixel_acc']:.4f} | F1: {baseline_val['mean_f1']:.4f}")
print(f"Test — mIoU: {baseline_test['mIoU']:.4f} | Acc: {baseline_test['pixel_acc']:.4f} | F1: {baseline_test['mean_f1']:.4f}")
print("\nPer-class IoU (baseline, test set):")
for cls, iou in sorted(baseline_test["per_class_iou"].items(), key=lambda x: -x[1]):
    print(f"  {cls:<30} {iou:.4f}")

In [ ]:
#model_ft, results_ft, test_ft = run_experiment(
 #   "segformer-b2-finetune-only",
 #   train_loader_base
#)

In [ ]:
#model_aug, results_aug, test_aug = run_experiment(
 #   "segformer-b2-finetune-augmented",
  #  train_loader_aug
#)

In [ ]:
def upsample_logits(logits, size=(512,512)):
    return F.interpolate(logits, size=size, mode="bilinear", align_corners=False)

def full_evaluate(model, loader):
    model.eval()
    all_miou, all_acc, all_f1 = [], [], []
    class_tp     = np.zeros(NUM_CLASSES - 1)
    class_fp     = np.zeros(NUM_CLASSES - 1)
    class_fn     = np.zeros(NUM_CLASSES - 1)

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(pixel_values=imgs)
            logits  = upsample_logits(outputs.logits)
            preds   = logits.argmax(dim=1).cpu().numpy()
            labels  = masks.cpu().numpy()

            for pred, label in zip(preds, labels):
                fp, fl = pred.flatten(), label.flatten()
                valid  = fl != IGNORE_INDEX
                fp, fl = fp[valid], fl[valid]
                if len(fl) == 0:
                    continue

                all_acc.append((fp == fl).sum() / len(fl))

                ious, f1s = [], []
                for cls in range(NUM_CLASSES - 1):
                    tp  = ((fp==cls) & (fl==cls)).sum()
                    fpp = ((fp==cls) & (fl!=cls)).sum()
                    fn  = ((fp!=cls) & (fl==cls)).sum()
                    class_tp[cls] += tp
                    class_fp[cls] += fpp
                    class_fn[cls] += fn
                    if tp + fpp + fn == 0:
                        continue
                    ious.append(tp / (tp + fpp + fn))
                    f1s.append((2*tp) / (2*tp + fpp + fn + 1e-8))

                if ious:
                    all_miou.append(np.mean(ious))
                    all_f1.append(np.mean(f1s))

    per_class_iou = {}
    per_class_f1  = {}
    for cls in range(NUM_CLASSES - 1):
        tp, fp, fn = class_tp[cls], class_fp[cls], class_fn[cls]
        if tp + fp + fn > 0:
            per_class_iou[ID2LABEL.get(cls, str(cls))] = tp / (tp + fp + fn)
            per_class_f1[ID2LABEL.get(cls, str(cls))]  = (2*tp) / (2*tp + fp + fn + 1e-8)

    return {
        "mIoU":          float(np.mean(all_miou)),
        "pixel_acc":     float(np.mean(all_acc)),
        "mean_f1":       float(np.mean(all_f1)),
        "per_class_iou": per_class_iou,
        "per_class_f1":  per_class_f1
    }

In [ ]:
FT_CHECKPOINT  = "/kaggle/input/models/mamadoubah8/segformer-b2-finetune-only/pytorch/default/1/segformer-b2-finetune-only_best.pt"
AUG_CHECKPOINT = "/kaggle/input/models/mamadoubah8/segformer-b2-finetune-augmented-best-pt/pytorch/default/1/segformer-b2-finetune-augmented_best.pt"
# ── Load checkpoints ──────────────────────────────────────────────────────────
def load_model(checkpoint_path=None):
    m = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/mit-b2", num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    )
    if checkpoint_path:
        m.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"Loaded: {checkpoint_path}")
    return m.to(device)

print("Evaluating baseline...")
baseline_model   = load_model()
baseline_test    = full_evaluate(baseline_model, test_loader)
del baseline_model; torch.cuda.empty_cache()

print("Evaluating fine-tune only...")
ft_model   = load_model(FT_CHECKPOINT)
test_ft    = full_evaluate(ft_model, test_loader)
del ft_model; torch.cuda.empty_cache()

print("Evaluating fine-tune + augmentation...")
aug_model  = load_model(AUG_CHECKPOINT)
test_aug   = full_evaluate(aug_model, test_loader)
del aug_model; torch.cuda.empty_cache()

# ── Results table ─────────────────────────────────────────────────────────────
key_classes = [
    "road", "person", "vegetation", "car", "sky",
    "building", "motorcycle", "autorickshaw", "animal", "sidewalk"
]

print("\n" + "="*65)
print("FINAL RESULTS — TEST SET")
print("="*65)
print(f"{'Method':<35} {'mIoU':>8} {'Acc':>8} {'F1':>8}")
print("-"*65)
print(f"{'Baseline (no adaptation)':<35} {baseline_test['mIoU']:>8.4f} {baseline_test['pixel_acc']:>8.4f} {baseline_test['mean_f1']:>8.4f}")
print(f"{'Fine-tune only':<35} {test_ft['mIoU']:>8.4f} {test_ft['pixel_acc']:>8.4f} {test_ft['mean_f1']:>8.4f}")
print(f"{'Fine-tune + Gambia augmentation':<35} {test_aug['mIoU']:>8.4f} {test_aug['pixel_acc']:>8.4f} {test_aug['mean_f1']:>8.4f}")

print("\nPer-class IoU — Gambian-relevant classes (test set):")
print(f"{'Class':<30} {'Baseline':>10} {'FT only':>10} {'FT+Aug':>10}")
print("-"*62)
for cls in key_classes:
    b  = baseline_test["per_class_iou"].get(cls, 0)
    ft = test_ft["per_class_iou"].get(cls, 0)
    ag = test_aug["per_class_iou"].get(cls, 0)
    print(f"{cls:<30} {b:>10.4f} {ft:>10.4f} {ag:>10.4f}")

# ── Full per-class table ──────────────────────────────────────────────────────
all_classes = sorted(
    set(baseline_test["per_class_iou"]) |
    set(test_ft["per_class_iou"]) |
    set(test_aug["per_class_iou"])
)
print("\nFull per-class IoU (all classes, sorted by FT only):")
print(f"{'Class':<30} {'Baseline':>10} {'FT only':>10} {'FT+Aug':>10}")
print("-"*62)
for cls in sorted(all_classes, key=lambda c: -test_ft["per_class_iou"].get(c, 0)):
    b  = baseline_test["per_class_iou"].get(cls, 0)
    ft = test_ft["per_class_iou"].get(cls, 0)
    ag = test_aug["per_class_iou"].get(cls, 0)
    print(f"{cls:<30} {b:>10.4f} {ft:>10.4f} {ag:>10.4f}")

# ── Learning curves ───────────────────────────────────────────────────────────
ft_miou  = [0.3889,0.4161,0.4241,0.4283,0.4400,0.4395,0.4481,0.4503,0.4524,0.4534,0.4595,0.4592,0.4591,0.4604,0.4609]
aug_miou = [0.3729,0.3865,0.4026,0.4135,0.4190,0.4274,0.4294,0.4342,0.4348,0.4412,0.4400,0.4400,0.4438,0.4448,0.4470]
ft_loss  = [0.5210,0.3626,0.3181,0.2894,0.2680,0.2482,0.2342,0.2206,0.2097,0.2012,0.1951,0.1883,0.1836,0.1797,0.1763]
aug_loss = [0.6793,0.4923,0.4460,0.4160,0.3948,0.3749,0.3629,0.3487,0.3373,0.3283,0.3178,0.3108,0.3054,0.3010,0.2962]
ft_f1    = [0.4630,0.4913,0.5004,0.5037,0.5164,0.5163,0.5257,0.5280,0.5300,0.5310,0.5374,0.5372,0.5370,0.5384,0.5389]
aug_f1   = [0.4462,0.4599,0.4769,0.4855,0.4945,0.5034,0.5049,0.5105,0.5112,0.5178,0.5167,0.5166,0.5205,0.5216,0.5243]
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_range, ft_miou,  marker='o', label="FT only")
axes[0].plot(epochs_range, aug_miou, marker='s', label="FT + Gambia Aug")
axes[0].axhline(baseline_test["mIoU"], color="red", linestyle="--", label="Baseline")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val mIoU")
axes[0].set_title("mIoU over epochs"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, ft_f1,  marker='o', label="FT only")
axes[1].plot(epochs_range, aug_f1, marker='s', label="FT + Gambia Aug")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1")
axes[1].set_title("F1 over epochs"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, ft_loss,  marker='o', label="FT only")
axes[2].plot(epochs_range, aug_loss, marker='s', label="FT + Gambia Aug")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Train Loss")
axes[2].set_title("Train loss over epochs"); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=150)
plt.show()

# ── Per-class bar chart ───────────────────────────────────────────────────────
b_vals   = [baseline_test["per_class_iou"].get(c, 0) for c in key_classes]
ft_vals  = [test_ft["per_class_iou"].get(c, 0)       for c in key_classes]
aug_vals = [test_aug["per_class_iou"].get(c, 0)      for c in key_classes]
x = np.arange(len(key_classes)); w = 0.25

fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x - w, b_vals,   w, label="Baseline",       color="#d62728")
ax.bar(x,     ft_vals,  w, label="FT only",         color="#1f77b4")
ax.bar(x + w, aug_vals, w, label="FT + Gambia Aug", color="#2ca02c")
ax.set_xticks(x); ax.set_xticklabels(key_classes, rotation=45, ha="right")
ax.set_ylabel("IoU"); ax.set_title("Per-class IoU — Gambian-relevant classes (SegFormer-B2)")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
plt.savefig("/kaggle/working/per_class_iou.png", dpi=150)
plt.show()

print("\nSaved: learning_curves.png, per_class_iou.png")

In [ ]:
import random

def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = img_tensor.permute(1,2,0).cpu().numpy()
    img  = img * std + mean
    return np.clip(img, 0, 1)

# Color palette for mask visualization
PALETTE = np.random.RandomState(42).randint(0, 255, (NUM_CLASSES, 3), dtype=np.uint8)
PALETTE[IGNORE_INDEX] = [0, 0, 0]

def colorize_mask(mask):
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for cls in range(NUM_CLASSES):
        color_mask[mask == cls] = PALETTE[cls]
    return color_mask

def get_failure_cases(model, dataset, n=6, seed=42):
    """Find samples where model prediction differs most from ground truth."""
    random.seed(seed)
    indices = random.sample(range(len(dataset)), min(100, len(dataset)))
    cases   = []

    model.eval()
    with torch.no_grad():
        for idx in indices:
            img, mask = dataset[idx]
            inp    = img.unsqueeze(0).to(device)
            output = model(pixel_values=inp)
            logits = upsample_logits(output.logits)
            pred   = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            gt     = mask.numpy()

            valid    = gt != IGNORE_INDEX
            if valid.sum() == 0:
                continue
            accuracy = (pred[valid] == gt[valid]).mean()
            cases.append((accuracy, img, gt, pred, idx))

    # Sort by worst accuracy
    cases.sort(key=lambda x: x[0])
    return cases[:n]

# ── Load fine-tune only model ─────────────────────────────────────────────────
ft_model = load_model(FT_CHECKPOINT)

print("Finding failure cases for fine-tune only model...")
ft_cases = get_failure_cases(ft_model, test_dataset, n=6)

fig, axes = plt.subplots(6, 3, figsize=(16, 24))
fig.suptitle("Failure Cases — Fine-tune Only (SegFormer-B2)", fontsize=14, y=1.01)

for row, (acc, img, gt, pred, idx) in enumerate(ft_cases):
    img_show = denormalize(img)
    gt_color   = colorize_mask(gt)
    pred_color = colorize_mask(pred)

    axes[row][0].imshow(img_show)
    axes[row][0].set_title(f"Image (sample {idx})")
    axes[row][0].axis("off")

    axes[row][1].imshow(gt_color)
    axes[row][1].set_title("Ground Truth")
    axes[row][1].axis("off")

    axes[row][2].imshow(pred_color)
    axes[row][2].set_title(f"Prediction (Acc: {acc:.2f})")
    axes[row][2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/failure_cases_ft.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: failure_cases_ft.png")

# ── Load augmented model ──────────────────────────────────────────────────────
del ft_model; torch.cuda.empty_cache()
aug_model = load_model(AUG_CHECKPOINT)

print("Finding failure cases for augmented model...")
aug_cases = get_failure_cases(aug_model, test_dataset, n=6)

fig, axes = plt.subplots(6, 3, figsize=(16, 24))
fig.suptitle("Failure Cases — Fine-tune + Gambia Aug (SegFormer-B2)", fontsize=14, y=1.01)

for row, (acc, img, gt, pred, idx) in enumerate(aug_cases):
    img_show   = denormalize(img)
    gt_color   = colorize_mask(gt)
    pred_color = colorize_mask(pred)

    axes[row][0].imshow(img_show)
    axes[row][0].set_title(f"Image (sample {idx})")
    axes[row][0].axis("off")

    axes[row][1].imshow(gt_color)
    axes[row][1].set_title("Ground Truth")
    axes[row][1].axis("off")

    axes[row][2].imshow(pred_color)
    axes[row][2].set_title(f"Prediction (Acc: {acc:.2f})")
    axes[row][2].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/failure_cases_aug.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: failure_cases_aug.png")
del aug_model; torch.cuda.empty_cache()

# ── Side by side comparison on same samples ───────────────────────────────────
ft_model  = load_model(FT_CHECKPOINT)
aug_model = load_model(AUG_CHECKPOINT)

print("Generating side-by-side comparison...")
sample_indices = [ft_cases[i][4] for i in range(4)]

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
fig.suptitle("Side-by-Side: FT only vs FT+Aug", fontsize=14, y=1.01)
cols = ["Image", "Ground Truth", "FT only Pred", "FT+Aug Pred"]
for ax, col in zip(axes[0], cols):
    ax.set_title(col, fontsize=11)

for row, idx in enumerate(sample_indices):
    img, mask = test_dataset[idx]
    inp = img.unsqueeze(0).to(device)

    with torch.no_grad():
        ft_pred  = upsample_logits(ft_model(pixel_values=inp).logits).argmax(1).squeeze(0).cpu().numpy()
        aug_pred = upsample_logits(aug_model(pixel_values=inp).logits).argmax(1).squeeze(0).cpu().numpy()

    img_show = denormalize(img)
    gt       = mask.numpy()

    axes[row][0].imshow(img_show);              axes[row][0].axis("off")
    axes[row][1].imshow(colorize_mask(gt));      axes[row][1].axis("off")
    axes[row][2].imshow(colorize_mask(ft_pred)); axes[row][2].axis("off")
    axes[row][3].imshow(colorize_mask(aug_pred));axes[row][3].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/comparison_ft_vs_aug.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: comparison_ft_vs_aug.png")

del ft_model, aug_model; torch.cuda.empty_cache()